# Hyperspectral pipeline with an EMIT image
Explore the EMIT hyperspectral cube → spectra → scatter plots → PCA → t-SNE → k-means → classification map.

The scene (EMIT, 17 Feb 2025, southern India) is in the Kaggle dataset **EMIT_60233**.
Data: Green, R. (2022). *EMIT L2A Estimated Surface Reflectance and Uncertainty and Masks 60 m V001*. NASA EOSDIS LP DAAC. https://doi.org/10.5067/EMIT/EMITL2ARFL.001

Runs on **Kaggle**, no NASA account needed. In the notebook editor: *Add Input* → search **EMIT_60233** → add it.


## 1. Install the extra packages and import libraries

In [ ]:
!pip install -q anywidget      
import numpy as np, matplotlib.pyplot as plt, xarray as xr
import plotly.express as px, plotly.graph_objects as go, ipywidgets as w
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

## 2. Load the EMIT scene from the Kaggle dataset and display it
Reflectance file (~1.8 GB) + mask file. The mask flags pixels you shouldn't trust (cloud, water, …).

In [ ]:
from glob import glob
scene = "20250217T082306_2504806_012"                                # date, time and ID in the file name
files = glob(f"/kaggle/input/**/*{scene}*.nc", recursive=True); print(files)
assert files, "No EMIT files found: add the EMIT_60233 dataset with 'Add Input' (right-hand panel)"
f      = [x for x in files if "_RFL_" in x][0]                          # reflectance file
mask_f = [x for x in files if "_MASK_" in x][0]                         # mask file

ds = xr.open_dataset(f)                                          # the reflectance cube + metadata
bp = xr.open_dataset(f, group="sensor_band_parameters")          # wavelengths + good-band flags
good = bp.good_wavelengths.values == 1                           # drop the water-vapour absorption bands
img  = ds.reflectance[:, :, good].values                         # (rows, cols, bands)
img.clip(0, 1, out=img)                                          # fill values (-9999) -> 0
wl   = bp.wavelengths.values[good]                               # nm
band = lambda nm: np.argmin(abs(wl - nm))                        # wavelength (nm) -> band index
stretch = lambda x: np.clip(x / np.percentile(x, 98), 0, 1)      # brighten an image for display

mk     = xr.open_dataset(mask_f)
mnames = list(xr.open_dataset(mask_f, group="sensor_band_parameters").mask_bands.values)
print("mask layers:", mnames)
flag   = mk["mask"].values[:, :, mnames.index("Aggregate Flag")]   # 1 = cloud / water / bad pixel

plt.figure(figsize=(14, 7))
plt.subplot(1, 2, 1); plt.imshow(stretch(img[:, :, [band(800), band(670), band(550)]])); plt.title("True colour")
plt.subplot(1, 2, 2); plt.imshow(flag, cmap="gray"); plt.title("Flagged pixels (white)")

## 3. Get to know the image cube
`img` is a 3-D NumPy array: **(rows, columns, bands)**. Every pixel `img[row, col, :]` is a full spectrum.

In [ ]:
print(f"number of rows    = {img.shape[0]}")
print(f"number of columns = {img.shape[1]}")
print(f"number of bands   = {img.shape[2]}")
print(wl)
ds          # the file's metadata

print("Minimum wavelength:", wl.min(), "nm")
print("Maximum wavelength:", wl.max(), "nm")
print("Total usable bands:", len(wl))

## 4. Display the image from three bands of your choice
Use band **indices** (0 … bands-1) or convert a wavelength with `band(nm)`. Try false colour: `band(860), band(650), band(560)`.

In [ ]:
#r, g, b = band(2200), band(1650), band(860)        # or plain indices, e.g. 35, 20, 10
#print(f"R = band {r} at {wl[r]:.1f} nm,  G = band {g} at {wl[g]:.1f} nm,  B = band {b} at {wl[b]:.1f} nm")

#plt.figure(figsize=(10, 10)); plt.imshow(stretch(img[:, :, [r, g, b]]))
# Modified SWIR-NIR false colour composite

# SWIR-NIR False Colour Composite
r, g, b = band(2200), band(1650), band(860)

print(
    f"Red: {wl[r]:.1f} nm | "
    f"Green: {wl[g]:.1f} nm | "
    f"Blue: {wl[b]:.1f} nm"
)

# Extract RGB bands
rgb = img[:, :, [r, g, b]].astype(float)

# Individual contrast enhancement for each channel
rgb_enhanced = np.zeros_like(rgb)

for i in range(3):
    channel = rgb[:, :, i]
    valid = (flag == 0) & np.isfinite(channel)

    p2, p98 = np.nanpercentile(channel[valid], [2, 98])

    rgb_enhanced[:, :, i] = np.clip(
        (channel - p2) / (p98 - p2 + 1e-10), 0, 1
    )

# Display flagged pixels in grey
rgb_enhanced[flag != 0] = [0.5, 0.5, 0.5]

# Plot composite and quality mask
fig, ax = plt.subplots(1, 2, figsize=(15, 7))

ax[0].imshow(rgb_enhanced)
ax[0].set_title(
    "Enhanced SWIR-NIR False Colour Composite\n"
    f"R: {wl[r]:.1f} nm | G: {wl[g]:.1f} nm | B: {wl[b]:.1f} nm"
)
ax[0].axis("off")

ax[1].imshow(flag, cmap="gray", vmin=0, vmax=1)
ax[1].set_title("EMIT Aggregate Quality Mask")
ax[1].axis("off")

plt.suptitle(
    "EMIT Hyperspectral Surface Visualization",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

## 5. Plot the spectrum of one pixel
⚠️ `img[row, col]` — **row is the y-axis, col is the x-axis**. The cross-hairs on the left show where the pixel is.

In [ ]:
# Section 5: Spectral signature of one pixel

# Original target pixel
target_row, target_col = 945, 597

# Find a valid pixel (Aggregate Flag = 0)
if flag[target_row, target_col] == 0:
    row, col = target_row, target_col
else:
    valid_pixels = np.argwhere(flag == 0)

    distances = np.sum(
        (valid_pixels - [target_row, target_col])**2,
        axis=1
    )

    row, col = valid_pixels[np.argmin(distances)]

print(f"Selected pixel: Row={row}, Column={col}")
print(f"Pixel flag: {flag[row, col]}")

# Extract spectral signature
spectrum = img[row, col, :]

# Exclude non-finite and zero-filled values
valid = np.isfinite(wl) & np.isfinite(spectrum) & (spectrum > 0)

# Use the same SWIR-NIR wavelengths as Section 4
r, g, b = band(2200), band(1650), band(860)

# Create two-panel visualization
fig, ax = plt.subplots(1, 2, figsize=(17, 7))

# Left: Image with selected pixel
ax[0].imshow(stretch(img[:, :, [r, g, b]]))
ax[0].axhline(row, color="red", linewidth=1.2)
ax[0].axvline(col, color="red", linewidth=1.2)
ax[0].scatter(col, row, color="yellow", marker="+", s=100)
ax[0].set_title("Selected Pixel Location")
ax[0].set_xlabel("Column")
ax[0].set_ylabel("Row")

# Right: Spectral signature
ax[1].plot(
    wl[valid],
    spectrum[valid],
    color="darkblue",
    linewidth=1.5
)

ax[1].set_title(
    f"EMIT Spectral Signature (Row={row}, Col={col})"
)
ax[1].set_xlabel("Wavelength (nm)")
ax[1].set_ylabel("Reflectance")
ax[1].grid(True, alpha=0.3)

plt.suptitle(
    "Pixel-wise Hyperspectral Analysis",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()
# Highlight SWIR region
ax[1].axvspan(
    2000, 2400,
    color="orange",
    alpha=0.12,
    label="SWIR mineral-sensitive region"
)

ax[1].legend()
ax[1].set_xlim(380, 2500)
ax[1].set_ylim(bottom=0)

ax[1].grid(True, linestyle="--", alpha=0.3)

## 6 Crop a small area and turn the cube into a table
The full scene has ~1.5 million pixels — too many for t-SNE. Pick a ~128×128 crop with a mix of land covers.
`reshape` turns the cube **(rows, cols, bands)** into a table **X = (pixels, bands)**: one row per pixel, one column per band.
This table is what every machine-learning tool below expects.

In [ ]:
row0, col0, size = 885, 533, 128
cube = img[row0:row0 + size, col0:col0 + size]
X    = cube.reshape(-1, cube.shape[2])                         # (pixels, bands)
# X = cube.reshape(size * size, cube.shape[2])     #alternative way to convert into table.
print(cube.shape, "->", X.shape); plt.imshow(stretch(cube[:, :, [band(2200), band(1650), band(860)]]))

## * 7. 2-D scatter plot: every pixel is a point, two bands are the axes


In [ ]:
#b1, b2 = band(650), band(860)
#plt.scatter(X[:, b1], X[:, b2], s=1)
#plt.xlabel(f"{wl[b1]:.0f} nm"); plt.ylabel(f"{wl[b2]:.0f} nm")
# Select two spectral bands
b1, b2 = band(860), band(2200)
b3 = band(1650)

# Remove flagged and invalid pixels
crop_flag = flag[row0:row0+size, col0:col0+size].ravel()

valid = (
    (crop_flag == 0) &
    np.isfinite(X).all(axis=1)
)

X_clean = X[valid]

# Reproducible sampling to reduce overlapping points
rng = np.random.default_rng(42)
n = min(6000, len(X_clean))
idx = rng.choice(len(X_clean), n, replace=False)

# Plot spectral relationship
plt.figure(figsize=(10, 8))

sc = plt.scatter(
    X_clean[idx, b1],
    X_clean[idx, b2],
    c=X_clean[idx, b3],
    cmap="viridis",
    s=3,
    alpha=0.35,
    edgecolors="none"
)

plt.colorbar(sc, label="Reflectance at 1650 nm")

plt.xlabel(f"Reflectance at {wl[b1]:.1f} nm")
plt.ylabel(f"Reflectance at {wl[b2]:.1f} nm")

plt.title("EMIT Spectral Band Scatter Plot")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Total valid pixels:", len(X_clean))
print("Pixels plotted:", n)

## 8. 3-D scatter plot (drag to rotate)

In [ ]:
#b1, b2, b3 = band(860), band(1650), band(2200)
#px.scatter_3d(x=X[:, b1], y=X[:, b2], z=X[:, b3], height=600,
              #labels=dict(x=f"{wl[b1]:.0f} nm", y=f"{wl[b2]:.0f} nm", z=f"{wl[b3]:.0f} nm")).update_traces(marker_size=1)
b1, b2, b3 = band(860), band(1650), band(2200)

fig = px.scatter_3d(
    x=X_clean[idx, b1],
    y=X_clean[idx, b2],
    z=X_clean[idx, b3],
    height=700,
    opacity=0.45,
    labels=dict(
        x=f"{wl[b1]:.1f} nm",
        y=f"{wl[b2]:.1f} nm",
        z=f"{wl[b3]:.1f} nm"
    )
)

fig.update_traces(marker=dict(size=2))
fig.update_layout(
    title="3D EMIT Spectral Reflectance Distribution",
    scene=dict(
        xaxis_title="NIR (860 nm)",
        yaxis_title="SWIR (1650 nm)",
        zaxis_title="SWIR (2200 nm)"
    )
)

fig.show()

## 8b. 100 random pixel spectra
Notice how the curves rise and fall together: the bands are far from independent.

In [ ]:
#n = 80
#idx = np.random.choice(len(X), n, replace=False)      # n random pixels from the table X
#plt.plot(wl, X[idx].T, alpha=0.25); plt.xlabel("Wavelength (nm)"); plt.ylabel("Reflectance")
rng = np.random.default_rng(42)

n = 80
idx = rng.choice(len(X_clean), n, replace=False)

plt.figure(figsize=(13, 7))

plt.plot(
    wl, X_clean[idx].T,
    alpha=0.20,
    linewidth=0.8
)

# Add median spectral signature
median_spectrum = np.median(X_clean[idx], axis=0)

plt.plot(
    wl, median_spectrum,
    color="red",
    linewidth=2.5,
    label="Median Spectrum"
)

plt.xlabel("Wavelength (nm)")
plt.ylabel("Reflectance")
plt.title("Spectral Variability of 80 EMIT Pixels")

plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 9. The data live on far fewer dimensions than the number of bands
We have ~240 bands (the *ambient* dimension), but neighbouring bands carry almost the same information.
Left: two adjacent bands fall on a straight line. Right: band-to-band correlation — large red blocks are groups of redundant bands.

In [ ]:
#plt.figure(figsize=(15, 7))
#plt.subplot(1, 2, 1); plt.scatter(X_clean[idx, band(860)], X_clean[idx, band(2200)], s=3,alpha=0.3); plt.title("860 nm vs 2200 nm")

#plt.subplot(1, 2, 2); plt.imshow(np.corrcoef(X_clean.T), cmap="coolwarm", vmin=-1, vmax=1, extent=[wl[0], wl[-1], wl[-1], wl[0]])
#plt.title("Band-to-band correlation"); plt.colorbar()
b1, b2 = band(860), band(2200)

idx = np.random.default_rng(42).choice(
    len(X_clean), min(12000, len(X_clean)), replace=False
)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].scatter(X_clean[idx, b1], X_clean[idx, b2],
              s=2, alpha=0.2, c='steelblue')
ax[0].set(xlabel=f'{wl[b1]:.1f} nm',
          ylabel=f'{wl[b2]:.1f} nm',
          title='860 nm vs 2200 nm')
ax[0].grid(alpha=0.3)

im = ax[1].imshow(np.corrcoef(X_clean.T),
                  cmap='coolwarm', vmin=-1, vmax=1)
ax[1].set_title('Band-to-Band Correlation')
ax[1].set(xlabel='Band Index', ylabel='Band Index')
fig.colorbar(im, ax=ax[1], label='Correlation')

plt.tight_layout()
plt.show()

## 10. PCA

In [ ]:
n_pc = 40
pca = PCA(n_components=n_pc).fit(X); P = pca.transform(X)       # P = (pixels, n_pc)

## 11. Scree plot
Move `elbow` to where you think the curve flattens. How many PCs out of ~240 bands do you really need?
Comment out `plt.yscale("log")` to see the difference.

In [ ]:
#elbow = 4 # Although we specify this value in the beginning, but it has to be diceded based on the output of the scree plot in the nexr cell's output.
#plt.plot(range(1, n_pc + 1), pca.explained_variance_ratio_, "o-")
#plt.yscale("log")
#plt.axvline(elbow, color="r", ls="--")
#plt.xlabel("PC"); plt.ylabel("Explained variance ratio")
# print(f"The first {elbow} PCs hold {pca.explained_variance_ratio_[:elbow].sum():.2%} of the variance of {X.shape[1]} bands")
elbow = 4

plt.plot(range(1, n_pc + 1), pca.explained_variance_ratio_, 'o-')
plt.yscale("log")
plt.axvline(elbow, color="r", ls="--")
plt.xlabel("PC")
plt.ylabel("Explained variance ratio")

print(f"Variance retained by first {elbow} PCs:",
      f"{pca.explained_variance_ratio_[:elbow].sum()*100:.2f}%")

plt.show()

## 12. Scatter two PCs against each other
PCs are numbered from 0. Also look at a PC as an image (right).

In [ ]:
pc_a, pc_b = 1, 2
plt.figure(figsize=(13, 5))
plt.subplot(1, 2, 1); plt.scatter(P[:, pc_a], P[:, pc_b], s=1); plt.xlabel(f"PC{pc_a}"); plt.ylabel(f"PC{pc_b}")
plt.subplot(1, 2, 2); plt.imshow(P[:, pc_a].reshape(size, size), cmap="RdBu_r"); plt.title(f"PC{pc_a} as an image")

## 13. t-SNE
You choose how many PCs go in, and the t-SNE settings. ~1–3 min for a 128×128 crop.
- `perplexity` (5–50): roughly how many neighbours each point "looks at"
- `early_exaggeration` (4–30): how strongly clusters are pulled apart early on
- `max_iter` (≥250), `learning_rate` ("auto" or 10–1000)

In [ ]:
n_pcs_used         = elbow       # how many PCs go into t-SNE (1 to n_pc)
perplexity         = 30          # roughly how many neighbours each point considers (5–50)
early_exaggeration = 12          # how hard clusters are pushed apart at the start (≥1; try 4–30)
learning_rate      = "auto"      # step size: "auto" or a number (10–1000)
max_iter           = 1200        # number of iterations (must be at least 250)
init               = "pca"    # starting layout: "pca" or "random"
random_state       = 42          # same number = same map every run; change it to see another layout

T = TSNE(n_components=2, perplexity=perplexity, early_exaggeration=early_exaggeration,
         learning_rate=learning_rate, max_iter=max_iter, init=init,
         random_state=random_state).fit_transform(P[:, :n_pcs_used])
plt.scatter(T[:, 0], T[:, 1], s=1)

## 14. k-means on the t-SNE map

In [ ]:
k = 5
labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(T)     # try .fit_predict(P[:, :n_pcs_used]) and compare!
plt.scatter(T[:, 0], T[:, 1], s=2, c=labels, cmap="tab10", vmin=0, vmax=k-1)

## 15. Original image vs classification map

In [ ]:
#plt.figure(figsize=(13, 6))
#plt.subplot(1, 2, 1); plt.imshow(stretch(cube[:, :, [band(650), band(560), band(470)]])); plt.title("True colour")
#plt.subplot(1, 2, 2); plt.imshow(labels.reshape(size, size), cmap="tab10", vmin=0, vmax=9, interpolation="nearest"); plt.title(f"k-means, k = {k}")
k = 5

labels = KMeans(
    n_clusters=k,
    n_init=20,
    random_state=42
).fit_predict(P[:, :n_pcs_used])

plt.figure(figsize=(10, 7))
plt.scatter(T[:, 0], T[:, 1], s=2, c=labels,
            cmap="tab10", vmin=0, vmax=k-1)

plt.title("K-means Classification on t-SNE Map")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster")
plt.tight_layout()
plt.show()

## 16. What is each class? Mean spectrum per cluster
Same colours as the map. Use the shapes (red edge, SWIR clay/water features, …) to name each class.

/kaggle/input/datasets/ranjansarkar/emit-60233/EMIT_L2A_MASK_001_20240502T034521_2412302_004.nc


In [ ]:
#for i in range(k): plt.plot(wl, X[labels == i].mean(0), color=plt.cm.tab10(i), label=f"class {i}")
#plt.legend(); plt.xlabel("Wavelength (nm)"); plt.ylabel("Mean reflectance")
plt.figure(figsize=(13, 6))

#colors = plt.cm.tab10.colors


#for i in range(k):
    
#mean_spectrum = np.nanmean(X[labels == i], axis=0)
    #plt.plot(wl, mean_spectrum,
             #color=colors[i],
             #linewidth=2.2,
             #alpha=0.95,
             #label=f"Class {i}")

#plt.xlabel("Wavelength (nm)", fontsize=12)
#plt.ylabel("Mean Reflectance", fontsize=12)
#plt.title("Mean Spectral Signature of 5 EMIT Clusters", fontsize=14)

#plt.legend(ncol=2, frameon=True)
#plt.grid(alpha=0.2)
#plt.tight_layout()
#plt.show()
plt.figure(figsize=(13, 6))

colors = plt.cm.tab10.colors

for i in range(k):
    spectra = X[labels == i]

    mean = np.nanmean(spectra, axis=0)
    std = np.nanstd(spectra, axis=0)

    plt.plot(wl, mean, color=colors[i],
             linewidth=2.2, label=f"Class {i}")

    plt.fill_between(wl, mean-std, mean+std,
                     color=colors[i], alpha=0.10)

    print(f"Class {i}: {len(spectra)} pixels")

plt.xlabel("Wavelength (nm)", fontsize=12)
plt.ylabel("Mean Reflectance", fontsize=12)
plt.title("Mean Spectral Signatures of 5 EMIT Clusters", fontsize=14)

plt.legend(ncol=2, frameon=True)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()